# Connect-4 Game Outcome Classification
## BITS Pilani - M.Tech (AIML) - Machine Learning Assignment 2

### 1. Problem Statement
The objective of this assignment is to develop, evaluate, and compare five supervised machine learning classification models on the **Connect-4 Opening Database** from the UCI Machine Learning Repository to predict the game outcome (`win`, `loss`, `draw`) for the first player (Player 1 / 'x') given an 8-ply legal board position.

### 2. Dataset Description
- **Source**: UCI Machine Learning Repository (ID: 26)
- **Instances**: 67,557 legal 8-ply positions
- **Features**: 42 categorical features corresponding to the 6x7 Connect-4 grid (columns `a` through `g`, rows `1` through `6`).
- **Values**: `x` (player x piece), `o` (player o piece), `b` (blank square).
- **Target**: `class` with 3 categories:
  - `win`: Player 1 will win (65.83%)
  - `loss`: Player 1 will lose (24.62%)
  - `draw`: Game ends in a draw (9.55%)

In [ ]:
import os
import json
import time
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, roc_auc_score, precision_score, recall_score,
    f1_score, matthews_corrcoef, classification_report, confusion_matrix
)

# Set plotting style
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

### 3. Data Loading and Exploration

In [ ]:
# Load test dataset or raw connect-4 data
df = pd.read_csv("test_data.csv")
X = df.iloc[:, :-1].astype(str)
y = df.iloc[:, -1].astype(str)

print(f"Test dataset shape: {df.shape}")
print("Class distribution:")
print(y.value_counts(normalize=True).apply(lambda v: f"{v*100:.2f}%"))
df.head()

### 4. Model Loading and Evaluation
We load the five pre-trained classification pipelines from `model/`:
1. **Logistic Regression**
2. **Decision Tree**
3. **K-Nearest Neighbors (KNN)**
4. **Naive Bayes (MultinomialNB)**
5. **Support Vector Machine (SVM - RBF Kernel)**

In [ ]:
models = {
    "Logistic Regression": joblib.load("model/logistic_regression.joblib"),
    "Decision Tree": joblib.load("model/decision_tree.joblib"),
    "KNN": joblib.load("model/knn.joblib"),
    "Naive Bayes": joblib.load("model/naive_bayes.joblib"),
    "Support Vector Machine": joblib.load("model/svm.joblib")
}

results = []
for name, pipe in models.items():
    t0 = time.time()
    y_pred = pipe.predict(X)
    y_proba = pipe.predict_proba(X)
    infer_time = time.time() - t0
    
    acc = accuracy_score(y, y_pred)
    auc = roc_auc_score(y, y_proba, multi_class="ovr", average="weighted")
    prec = precision_score(y, y_pred, average="weighted", zero_division=0)
    rec = recall_score(y, y_pred, average="weighted", zero_division=0)
    f1 = f1_score(y, y_pred, average="weighted", zero_division=0)
    mcc = matthews_corrcoef(y, y_pred)
    
    results.append({
        "ML Model Name": name,
        "Accuracy": round(acc, 4),
        "AUC": round(auc, 4),
        "Precision": round(prec, 4),
        "Recall": round(rec, 4),
        "F1": round(f1, 4),
        "MCC": round(mcc, 4)
    })

comparison_df = pd.DataFrame(results)
display(comparison_df)

### 5. Metric Comparison and Visualization

In [ ]:
# Plot evaluation metrics comparison
plot_df = comparison_df.melt(id_vars="ML Model Name", value_vars=["Accuracy", "AUC", "Precision", "Recall", "F1", "MCC"], var_name="Metric", value_name="Score")

plt.figure(figsize=(12, 6))
sns.barplot(data=plot_df, x="ML Model Name", y="Score", hue="Metric", palette="viridis")
plt.title("Comparison of 5 Classification Models on Connect-4 Dataset", fontsize=14, fontweight="bold")
plt.ylim(0, 1.0)
plt.ylabel("Score")
plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()
plt.show()

### 6. Confusion Matrix for the Winning Model (Support Vector Machine)

In [ ]:
winner = models["Support Vector Machine"]
y_pred_svm = winner.predict(X)
classes = list(winner.classes_)
cm = confusion_matrix(y, y_pred_svm, labels=classes)

plt.figure(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=classes, yticklabels=classes)
plt.title("Confusion Matrix - Support Vector Machine", fontsize=13, fontweight="bold")
plt.xlabel("Predicted Class")
plt.ylabel("Actual Class")
plt.tight_layout()
plt.show()

print("Classification Report:")
print(classification_report(y, y_pred_svm, target_names=classes))

### 7. Key Observations and Performance Discussion
- **Support Vector Machine (SVM)** is the overall winner with **Accuracy: 79.43%**, **AUC: 0.9066**, **Precision: 77.64%**, **Recall: 79.43%**, **F1: 75.93%**, and **MCC: 0.5521**.
- The non-linear RBF kernel maps board positions into a high-dimensional reproducing kernel Hilbert space where geometric alignments (horizontal, vertical, diagonal 4-in-a-row connections) become linearly separable.
- **Logistic Regression** and **KNN** provide strong baselines with ~75.5% accuracy.
- **Decision Tree** achieves 71.5% accuracy; while interpretable, individual axis-aligned splits struggle with multi-directional connect-4 tactical combinations without tree ensemble methods.
- **Naive Bayes** achieves 72.1% accuracy; its strict conditional independence assumption between grid squares is violated by the contiguous nature of Connect-4 pieces.